# Initiating observation via Phoenix

In [1]:
# %pip install arize-phoenix
# %pip install llama-index-callbacks-arize-phoenix
# observability
import llama_index.core  # type: ignore
import phoenix as px  # type: ignore

px.launch_app()

llama_index.core.set_global_handler("arize_phoenix", endpoint="http://localhost:6006/v1/traces")

🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://docs.arize.com/phoenix


# Testing retriever by hand

# Testing retriever with eval framework

## Initializing Test Suite

### Programmatically instantiating a test suite 

In [1]:
from evidence_seeker_eval import (
    RetrievalTestSuite,
    QueryTestCase,
    DocumentRelevance,
    CriterionType,
    TestQuery,
    RetrievalPipelineConfig,
)
from evidence_seeker import (
    RetrievalConfig,
)

# model kwargs for the LLM-Judge used to evaluate relevance criteria for 
# each document
model_kwargs = {
    "name": "Llama-3.3-70B-Instruct",
    "description": "Llama-3.3-70B served by Together.ai over HuggingFace",
    "base_url": "https://router.huggingface.co/v1",
    "model": "meta-llama/Llama-3.3-70B-Instruct:together",
    "model_link": "https://huggingface.co/meta-llama/Llama-3.3-70B-Instruct",
    "api_key_name": "hf_debatelab_inference_provider",
    "backend_type": "openai",
    "default_headers": 
        {"X-HF-Bill-To": "DebateLabKIT"},
    "max_tokens": 1024,
    "temperature": 0.2,
    "timeout": 260,
}
# just as an example of configuring one particular retriever that 
# is being tested
retrieval_config = RetrievalConfig.from_config_file(
    "../configs/retriever_config.yaml"
)
retrieval_pipeline_config = RetrievalPipelineConfig(
    key="retrieval_setup_01",
    pipeline_key="default_retrieval_pipeline",
    name="test name",
    **retrieval_config.model_dump(),
)

# Define specific criteria to evaluate individual documents 
criteria = [
    DocumentRelevance(
        name="direct_mention",
        description="Query terms are directly mentioned in document",
        criterion_type=CriterionType.BINARY,
        prompt_template="""
        Query: "{query}"
        
        Document: "{document}"
        
        Question: Are the main terms from the query directly mentioned in this document?
        Answer only: Yes or No
        """,
        expected_values=[0, 1],
        threshold=1
    ),
]
test_queries = {
    "query_01": TestQuery(
        query= "Carl Schmitt war der Meinung, dass liberale Demokratien nicht in der Lage sind, sich gegen ihre Feinde zu verteidigen.",
        comment="Fastzitate (= leichte Paraphrasierungen) aus APuz 9-11/24, S. 26",
        test_cases = [
            "min_pass_1",
            "one_pass_from_hacke",
            QueryTestCase(  # an inline test case
                name="one_from_hacke",
                test_case_type="min_docs_contain_metadata",
                description=(
                    "Test case to check whether we found at least "
                    "one doc from the author 'Jens Hacke' "
                    "irrespective of any criterion."  # no criterion here!
                ),
                kwargs={
                    'min_documents': 1,
                    'meta_data_match': {
                        'author': 'Hacke, Jens',
                        'file_name': 'hacke_wehrhafte_2024.pdf',
                    },
                },
            ),
            QueryTestCase(  # an inline test case
                name="one_doc_contains_carl_schmitt",
                test_case_type="min_docs_contain_text",
                description=(
                    "Test case to check whether we found at least "
                    "one doc from that (fuzzy-)contains the given "
                    "needle irrespective of any criterion."  # no criterion here!
                ),
                kwargs={
                    'min_documents': 1,
                    'needle': ( # needle with some typos to test fuzzy matching
                        "Ders Staatssrechtler Carlos[sic!] Schmitt etwa meinte die "
                        "Schwwachstelle des Liberalismus darin zu erkennen"
                    ),
                    'threshold': 80,  # fuzzy matching threshold
                },
            ),
        ]
    )
}

test_suite = RetrievalTestSuite(
    uid="retriever_test_suite_example",
    used_llm_judge="Llama-3.3-70B-Instruct",
    llm_judges={"Llama-3.3-70B-Instruct": model_kwargs},
    # retrievers=[retrieval_config],
    retrievers={
        retrieval_pipeline_config.key: retrieval_pipeline_config,
    },
    relevance_criteria=criteria,
    test_cases=[
        QueryTestCase(
            name="min_pass_1",
            test_case_type="min_pass",
            description= (
                "TestCase that checks whether at least "
                "one document from the retrieved documents "
                "satisifies the `direct_mention` criterion."
            ),
            criterion_name="direct_mention",
            kwargs={'min_documents': 1},
        ),
        QueryTestCase(
            name="one_pass_from_hacke",
            test_case_type="min_docs_contain_metadata",
            description=(
                "Test case to check whether we found at least "
                "one doc from the author 'Jens Hacke' that "
                "satisfies the `direct_mention` criterion."
            ),
            criterion_name="direct_mention",
            kwargs={
                'min_documents': 1,
                'meta_data_match': {
                    'author': 'Hacke, Jens',
                    'file_name': 'hacke_wehrhafte_2024.pdf',
                },
            },
        ),
        
    ],
    test_queries=test_queries,
)

test_suite.to_yaml_file(f"../test_suites/retriever/{test_suite.uid}.yaml")
#test_suite.to_yaml_files("../test_suites/retriever")


/home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker-eval/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-06 11:12:15.217 | WARNING  | evidence_seeker.retrieval.config:check_hub_token_name:137 - Check whether you need a HF hub token for saving/loading your index to/from the Hugging Face Hub. If you need one, set an `hub_key_name` in the retriever config and provide the token as an environment variable with that name.
2025-10-06 11:12:15.218 | WARNING  | evidence_seeker.retrieval.config:load_env_file:188 - No environment file with API keys/passwords specified for retriever. Please set 'env_file' to a valid path if you want to load environment variables from a file.
2025-10-06 11:12:15.218 | WARNING  | evidence_seeker.retrieval.config:check_hub_token_name:137 - 

### Loading Test Suite from YAML

In [1]:
# loading test suite from ONE file
from evidence_seeker_eval import (
    RetrievalTestSuite
)
test_suite = RetrievalTestSuite.from_yaml_file("../test_suites/retriever/retriever_test_suite_example.yaml")
# test_suite = RetrievalTestSuite.from_yaml_file("../test_suites/retriever/retriever_test_suite_01.yaml")
#test_suite.to_yaml_files("../test_suites/retriever")

/home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker-eval/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-09-03 19:51:47.501 | WARNING  | evidence_seeker.retrieval.config:check_hub_token_name:119 - Check whether you need a HF hub token for saving/loading your index to/from the Hugging Face Hub. If you need one, set an `hub_key_name` in the retriever config and provide the token as an environment variable with that name.
2025-09-03 19:51:47.502 | WARNING  | evidence_seeker.retrieval.config:load_env_file:145 - No environment file with API keys specified for retriever. Please set 'env_file' to a valid path if you want to load environment variables from a file.


In [2]:
from evidence_seeker_eval import (
    RetrievalTestSuite
)
# loading test suite from MULTIPLE files
test_suite = RetrievalTestSuite.from_yaml_files(
    base_dir="../test_suites/retriever",
    #base_name="ts_01",
    # meta_data_file_name="ts_index_creation_metadata.yaml",
    # meta_data_file_name="ts_models_metadata.yaml",
    # meta_data_file_name="ts_topk_metadata.yaml",
    meta_data_file_name="ts_query_strategies_metadata.yaml",
    test_cases_file_name="test_cases.yaml",
    queries_file_name="queries.yaml",
    # test_cases_file_name="one_test_case.yaml",
    # queries_file_name="one_query.yaml",
)


/home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker-eval/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-07 11:47:27.963 | WARNING  | evidence_seeker.retrieval.config:check_hub_token_name:137 - Check whether you need a HF hub token for saving/loading your index to/from the Hugging Face Hub. If you need one, set an `hub_key_name` in the retriever config and provide the token as an environment variable with that name.
2025-10-07 11:47:27.966 | INFO     | evidence_seeker.retrieval.config:load_env_file:208 - Loaded environment variables from '../.env'
2025-10-07 11:47:27.967 | WARNING  | evidence_seeker.retrieval.config:check_hub_token_name:137 - Check whether you need a HF hub token for saving/loading your index to/from the Hugging Face Hub. If you need one, set

## 🚧 Example: Adding additional test functions for test cases

In [ ]:
...

## Running Tests

In [3]:
import pathlib
import yaml

from evidence_seeker_eval import (
    RetrievalTestSuite,
    LLMBasedRetrievalEvaluator,
    RetrievalEvaluationResult,
)

# Initialize LLM judge
evaluator = LLMBasedRetrievalEvaluator(
    test_suite=test_suite
)

# Run evaluation
#results = asyncio.run(run_llm_based_evaluation())
#results = await evaluator.evaluate_embedding_models()
result: RetrievalEvaluationResult = await evaluator()


2025-10-07 11:49:10.418 | DEBUG    | evidence_seeker.backend:get_openai_llm:233 - Fetching api key via env var: hf_debatelab_inference_provider
2025-10-07 11:49:10.419 | DEBUG    | evidence_seeker.backend:get_openai_llm:246 - Instantiating OpenAILike model (model: meta-llama/Llama-3.3-70B-Instruct:together,base_url: https://router.huggingface.co/v1).
2025-10-07 11:49:10.419 | DEBUG    | evidence_seeker.retrieval.base:_get_embed_model:236 - Inititializing embed model: BAAI/bge-m3 
2025-10-07 11:49:13.553 | DEBUG    | evidence_seeker.retrieval.base:_get_embedding_dimension:294 - Determined embed_dim by sampling: 768
2025-10-07 11:49:13.554 | INFO     | evidence_seeker.retrieval.document_retriever:load_index:105 - Using index persist path: /home/basti/tmp/APUZ/storage/paraphrase-multilingual-mpnet-base-v2
2025-10-07 11:49:13.554 | INFO     | evidence_seeker.retrieval.document_retriever:load_index:137 - Loading index from disk at /home/basti/tmp/APUZ/storage/paraphrase-multilingual-mpnet-b

Loading llama_index.core.storage.kvstore.simple_kvstore from /home/basti/tmp/APUZ/storage/paraphrase-multilingual-mpnet-base-v2/index/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from /home/basti/tmp/APUZ/storage/paraphrase-multilingual-mpnet-base-v2/index/index_store.json.


2025-10-07 11:50:30.009 | DEBUG    | evidence_seeker.retrieval.base:_get_embed_model:236 - Inititializing embed model: BAAI/bge-m3 
2025-10-07 11:50:32.299 | DEBUG    | evidence_seeker.retrieval.base:_get_embedding_dimension:294 - Determined embed_dim by sampling: 768
2025-10-07 11:50:32.300 | INFO     | evidence_seeker.retrieval.document_retriever:load_index:105 - Using index persist path: /home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker/TMP/APUZ_PART_POPULISMUS/storage/paraphrase-multilingual-mpnet-base-v2
2025-10-07 11:50:32.300 | INFO     | evidence_seeker.retrieval.document_retriever:load_index:137 - Loading index from disk at /home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker/TMP/APUZ_PART_POPULISMUS/storage/paraphrase-multilingual-mpnet-base-v2/index


Loading llama_index.core.storage.kvstore.simple_kvstore from /home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker/TMP/APUZ_PART_POPULISMUS/storage/paraphrase-multilingual-mpnet-base-v2/index/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from /home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker/TMP/APUZ_PART_POPULISMUS/storage/paraphrase-multilingual-mpnet-base-v2/index/index_store.json.


2025-10-07 11:50:35.253 | DEBUG    | evidence_seeker.retrieval.base:_get_embed_model:236 - Inititializing embed model: BAAI/bge-m3 
2025-10-07 11:50:38.051 | DEBUG    | evidence_seeker.retrieval.base:_get_embedding_dimension:294 - Determined embed_dim by sampling: 768
2025-10-07 11:50:38.052 | INFO     | evidence_seeker.retrieval.document_retriever:load_index:105 - Using index persist path: /home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker/TMP/APUZ_PART_POPULISMUS/storage/paraphrase-multilingual-mpnet-base-v2
2025-10-07 11:50:38.053 | INFO     | evidence_seeker.retrieval.document_retriever:load_index:137 - Loading index from disk at /home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker/TMP/APUZ_PART_POPULISMUS/storage/paraphrase-multilingual-mpnet-base-v2/index


Loading llama_index.core.storage.kvstore.simple_kvstore from /home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker/TMP/APUZ_PART_POPULISMUS/storage/paraphrase-multilingual-mpnet-base-v2/index/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from /home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker/TMP/APUZ_PART_POPULISMUS/storage/paraphrase-multilingual-mpnet-base-v2/index/index_store.json.


2025-10-07 11:50:40.688 | DEBUG    | evidence_seeker.retrieval.base:_get_embed_model:236 - Inititializing embed model: BAAI/bge-m3 
2025-10-07 11:50:43.976 | DEBUG    | evidence_seeker.retrieval.base:_get_embedding_dimension:294 - Determined embed_dim by sampling: 768
2025-10-07 11:50:43.977 | INFO     | evidence_seeker.retrieval.document_retriever:load_index:105 - Using index persist path: /home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker/TMP/APUZ_PART_POPULISMUS/storage/paraphrase-multilingual-mpnet-base-v2
2025-10-07 11:50:43.977 | INFO     | evidence_seeker.retrieval.document_retriever:load_index:137 - Loading index from disk at /home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker/TMP/APUZ_PART_POPULISMUS/storage/paraphrase-multilingual-mpnet-base-v2/index


Loading llama_index.core.storage.kvstore.simple_kvstore from /home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker/TMP/APUZ_PART_POPULISMUS/storage/paraphrase-multilingual-mpnet-base-v2/index/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from /home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker/TMP/APUZ_PART_POPULISMUS/storage/paraphrase-multilingual-mpnet-base-v2/index/index_store.json.


2025-10-07 11:50:46.322 | INFO     | evidence_seeker_eval.retriever.base:evaluate_embedding_models:302 - 
=== Evaluating mpnet_base_pop_evse_v0.1.1 ===
2025-10-07 11:50:46.322 | INFO     | evidence_seeker_eval.retriever.base:evaluate_embedding_model:276 - Evaluating query query_01: Carl Schmitt war der Meinung, dass liberale Demokr...
2025-10-07 11:50:47.487 | INFO     | evidence_seeker_eval.retriever.base:_evaluate_by_criterion:153 - Evaluating criterion: direct_mention
2025-10-07 11:50:58.938 | INFO     | evidence_seeker_eval.retriever.base:_evaluate_by_criterion:153 - Evaluating criterion: binary_relevance
2025-10-07 11:51:04.879 | INFO     | evidence_seeker_eval.retriever.base:_evaluate_by_criterion:153 - Evaluating criterion: scaled_relevance
2025-10-07 11:51:12.044 | INFO     | evidence_seeker_eval.retriever.base:_evaluate_by_criterion:153 - Evaluating criterion: direct_quote
2025-10-07 11:51:18.400 | INFO     | evidence_seeker_eval.retriever.base:evaluate_embedding_model:276 - E

### Saving Test Results

In [4]:
from datetime import datetime
timestamp = datetime.strptime(result.time, "%Y-%m-%d %H:%M:%S UTC").strftime("%Y-%m-%d_%H-%M-%S")
yaml_file_name = (
    f"{result.test_suite.uid}"
    f"_{timestamp}"
    "_eval.yaml"
)
yaml_file = pathlib.Path(f"../evaluation_results/{yaml_file_name}")
result.to_yaml_file(yaml_file)


### Displaying Results

In [5]:
from pathlib import Path
from evidence_seeker_eval.retriever.util import create_retriever_eval_report

# yaml_file = "../evaluation_results/retriever_eval_2025-08-28T14-03-58.yaml"
# yaml_file = Path(f"../evaluation_results/retriever_eval_{timestamp}.yaml"
#output_file = "reports/evaluation_report.md"
template_file = Path("../templates/retriever_report_tmpl.md")
md_output = create_retriever_eval_report(
    yaml_file_path=yaml_file, 
    template=template_file,
    save_md=True,
    output_dir="../reports",
)
# display md content in a Jupyter notebook
from IPython.display import Markdown, display
display(Markdown(md_output))

2025-10-07 14:59:13.125 | WARNING  | evidence_seeker.retrieval.config:check_hub_token_name:137 - Check whether you need a HF hub token for saving/loading your index to/from the Hugging Face Hub. If you need one, set an `hub_key_name` in the retriever config and provide the token as an environment variable with that name.
2025-10-07 14:59:13.126 | INFO     | evidence_seeker.retrieval.config:load_env_file:208 - Loaded environment variables from '../.env'
2025-10-07 14:59:13.127 | WARNING  | evidence_seeker.retrieval.config:check_hub_token_name:137 - Check whether you need a HF hub token for saving/loading your index to/from the Hugging Face Hub. If you need one, set an `hub_key_name` in the retriever config and provide the token as an environment variable with that name.
2025-10-07 14:59:13.127 | INFO     | evidence_seeker.retrieval.config:load_env_file:208 - Loaded environment variables from '../.env'
2025-10-07 14:59:13.128 | WARNING  | evidence_seeker.retrieval.config:check_hub_token_

Evaluation report saved to: ../reports/ts_query_strategies_mpnet_2025-10-07_09-50-46_eval_report.md


---
source_file: ts_query_strategies_mpnet_2025-10-07_09-50-46_eval.yaml
evaluation_time: 2025-10-07 09:50:46 UTC
test_suite_uid: ts_query_strategies_mpnet
---

# Evaluation Report: 75838989-3b0a-4ce3-a982-24622623da8b

This evaluation report was generated from: `ts_query_strategies_mpnet_2025-10-07_09-50-46_eval.yaml`


## Test Suite Metadata

*Description of the test suite and its configurations:* Test suite based on the full APuZ example dataset.
Index construction based on EvSe v0.1.1 with paraphrase-multilingual-mpnet-base-v2 
as embedding model.
We vary different strategies to retrieve documents (EvSe v0.1.1 retrieval as baseline, query decomposition and hybrid search).


- *Test Suite Name/Uid*: ts_query_strategies_mpnet 
- *Evaluation Time*: 2025-10-07 09:50:46 UTC
- *Result UID*: 75838989-3b0a-4ce3-a982-24622623da8b
- *LLM Judge*: Llama-3.3-70B-Instruct
- *Status*: completed
- *Source File*: ts_query_strategies_mpnet_2025-10-07_09-50-46_eval.yaml
- *n Retrievers*: 4
- *n Queries*: 32
- *n Criteria*: 4
- *n Test Cases*: 24

### Retriever Configurations



**Retriever pipeline: `mpnet_base_pop_evse_v0.1.1`**

- *Embed Model*: BAAI/bge-m3
- *Top-k*: 8
- *Window Size*: 3

Baseline: Both index construction and retrieval is based one evidence-seeker v0.1.1.




**Retriever pipeline: `mpnet_base_pop_evse_v0.1.1_hybrid_search`**

- *Embed Model*: BAAI/bge-m3
- *Top-k*: 8
- *Window Size*: 3

The index is constructed with evidence-seeker v0.1.1 but uses a hybrid search (implemented in `HybridRetrievalPipeline`):

HybridRetrievalPipeline implements a hybrid document retrieval strategy
that combines vector-based search with BM25 ranking and Reciprocal Rank
Fusion (RRF) to improve retrieval quality.

  + We retrieve many documents with a simple vector search (`top_k_init`).
  + We use an LLM to produce a list of relevant terms from the user query (including synonyms).
    + We use meta-llama/Llama-3.2-3B-Instruct-Turbo as LLM.
  + We use BM25 (from the `rank_bm25` package) to calculate relevance scores for the retrieved documents (whole window) based on these terms.
  + We aggregate both relevance scores (vector search and BM25) with reciprocal rank fusion to get a final ranking.
  + We return the `top_k` documents ($top_k<top_{k_{init}}$).




**Retriever pipeline: `mpnet_base_pop_evse_v0.1.1_query_decomposition`**

- *Embed Model*: BAAI/bge-m3
- *Top-k*: 8
- *Window Size*: 3

The index is constructed with evidence-seeker v0.1.1 but we query by 
query decomposition: An LLM (meta-llama/Llama-3.2-3B-Instruct-Turbo) is 
tasked to provide alternative formulations
of the input query, then we query w.r.t. all query alternatives and merge
them (ranked by their frequency-weighted relevance score) 
such that we have top_k results.




**Retriever pipeline: `mpnet_base_pop_evse_v0.1.1_simple_hybrid_search`**

- *Embed Model*: BAAI/bge-m3
- *Top-k*: 8
- *Window Size*: 3

The index is constructed with evidence-seeker v0.1.1 but uses a hybrid search
(implemented in `SimpleHybridRetrievalPipeline`):

SimpleHybridRetrievalPipeline implements a hybrid document retrieval strategy
that combines vector-based search with BM25 ranking and Reciprocal Rank
Fusion (RRF) to improve retrieval quality.

  + We retrieve many documents with a simple vector search (`top_k_init`).
  + We split the input query on spaces to produce a list of relevant search terms from the user query.
  + We use BM25 (from the `rank_bm25` package) to calculate relevance scores for the retrieved documents (whole window) based on these terms.
  + We aggregate both relevance scores (vector search and BM25) with reciprocal rank fusion to get a final ranking.
  + We return the `top_k` documents ($top_k<top_{k_{init}}$).





## Testresults


### Retrieval pipeline: mpnet_base_pop_evse_v0.1.1

**Test Case Performance:**



Overall pass rate: 41 out of 81 tests passed (50.6%)

Pass rates by test case name: 


- **min_pass_1**: 9 out of 10 tests passed (90.0%)


- **min_pass_2**: 1 out of 1 tests passed (100.0%)


- **one_from_hacke**: 1 out of 1 tests passed (100.0%)


- **one_from_baer**: 0 out of 1 tests passed (0.0%)


- **one_from_diehl**: 7 out of 7 tests passed (100.0%)


- **one_from_merkel**: 2 out of 3 tests passed (66.7%)


- **one_from_pickel**: 1 out of 1 tests passed (100.0%)


- **min_pass_direct_mention_1**: 6 out of 15 tests passed (40.0%)


- **min_pass_binary_relevance_3**: 9 out of 12 tests passed (75.0%)


- **one_from_baer_marandi**: 0 out of 4 tests passed (0.0%)


- **min_pass_direct_mention_3**: 0 out of 6 tests passed (0.0%)


- **exact_pass_direct_quote_1**: 0 out of 3 tests passed (0.0%)


- **min_percentage_60_scaled_relevance**: 0 out of 6 tests passed (0.0%)


- **test_case_9061e358-a8f2-47b1-9360-40c671de2faa**: 1 out of 1 tests passed (100.0%)


- **test_case_c8689144-3876-416d-a511-3d2d5945b511**: 1 out of 1 tests passed (100.0%)


- **test_case_2073fd93-dcb1-43bc-9977-bb814f5dec94**: 0 out of 1 tests passed (0.0%)


- **test_case_ba637ae3-20b0-4c7e-9483-d52b278b49e3**: 0 out of 1 tests passed (0.0%)


- **test_case_d4ce0241-a805-44cc-bf8f-1f9b15dcf053**: 1 out of 1 tests passed (100.0%)


- **test_case_6f7514a2-67af-4d3c-8624-f55d37d8f402**: 0 out of 1 tests passed (0.0%)


- **test_case_353142a0-0df4-448f-972d-9a373435cc0f**: 1 out of 1 tests passed (100.0%)


- **test_case_777d34cf-39bb-4ed5-9f0b-aa48492b96b1**: 0 out of 1 tests passed (0.0%)


- **test_case_f2915813-eeb1-4ab4-b775-62560d30b816**: 1 out of 1 tests passed (100.0%)


- **test_case_c8da6d94-2684-496a-8423-a2d579d1f0fa**: 0 out of 1 tests passed (0.0%)


- **test_case_bbc4773f-44eb-4f9a-9220-da021f7c6eb8**: 0 out of 1 tests passed (0.0%)


**Relevance Criteria Performance:**


- **direct_mention**: 48 out of 256 documents satisfied (18.8%)

- **binary_relevance**: 119 out of 256 documents satisfied (46.5%)

- **scaled_relevance**: 34 out of 256 documents satisfied (13.3%)

- **direct_quote**: 2 out of 256 documents satisfied (0.8%)



<details>
<summary>Expand/collapse query-specific details</summary>


+ Query: query_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 0.5000
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_hacke**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_9061e358-a8f2-47b1-9360-40c671de2faa**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.8750, Mean Score = 0.8750
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.3750, Mean Score = 3.0000
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_c8689144-3876-416d-a511-3d2d5945b511**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer**: ❌ FAILED (None docs passed relevance criterion)
        
        - **test_case_2073fd93-dcb1-43bc-9977-bb814f5dec94**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_04
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.7500
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_merkel**: ❌ FAILED (None docs passed relevance criterion)
        
        - **test_case_ba637ae3-20b0-4c7e-9483-d52b278b49e3**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_05
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.1250, Mean Score = 0.1250
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.3750
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_pickel**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_d4ce0241-a805-44cc-bf8f-1f9b15dcf053**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_06
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_mention**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.3750, Mean Score = 2.8750
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (4 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_6f7514a2-67af-4d3c-8624-f55d37d8f402**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_07
    + Criteria Results:
        
        - **binary_relevance**: Precision = 1.0000, Mean Score = 1.0000
        
        - **direct_mention**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 2.8750
        
    + Test Case Results:
        
        - **min_pass_2**: ✅ PASSED (5 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_353142a0-0df4-448f-972d-9a373435cc0f**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_08
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_mention**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.1250
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (6 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_777d34cf-39bb-4ed5-9f0b-aa48492b96b1**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_09
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.8750, Mean Score = 0.8750
        
        - **direct_mention**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.3750
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (6 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_f2915813-eeb1-4ab4-b775-62560d30b816**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_10
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.8750, Mean Score = 0.8750
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.6250
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (2 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_c8da6d94-2684-496a-8423-a2d579d1f0fa**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_11
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 2.7500
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (2 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_bbc4773f-44eb-4f9a-9220-da021f7c6eb8**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_ascriptions_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.8750
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (4 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ❌ FAILED (4 docs passed relevance criterion)
        

+ Query: query_ascriptions_01_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 1.1250
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (2 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ❌ FAILED (2 docs passed relevance criterion)
        

+ Query: query_ascriptions_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_ascriptions_02_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_ascriptions_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_mention**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 2.2500
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (3 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ✅ PASSED (4 docs passed relevance criterion)
        
        - **one_from_merkel**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_ascriptions_03_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.1250
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (4 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ✅ PASSED (2 docs passed relevance criterion)
        
        - **one_from_merkel**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_paraphrase_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 1.7500
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (3 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (1 docs passed relevance criterion)
        

+ Query: query_paraphrase_01_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (3 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.5000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (4 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_02_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 1.7500
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (4 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 1.0000, Mean Score = 1.0000
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 3.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (8 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (2 docs passed relevance criterion)
        

+ Query: query_paraphrase_03_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.8750, Mean Score = 0.8750
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.8750
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (7 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (1 docs passed relevance criterion)
        

+ Query: query_quotes_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 1.0000, Mean Score = 1.0000
        
        - **direct_mention**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 3.1250
        
    + Test Case Results:
        
        - **exact_pass_direct_quote_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ✅ PASSED (6 docs passed relevance criterion)
        

+ Query: query_quotes_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.7500
        
    + Test Case Results:
        
        - **exact_pass_direct_quote_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_quotes_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 2.1250
        
    + Test Case Results:
        
        - **exact_pass_direct_quote_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_relevance_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.1250, Mean Score = 0.1250
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 0.5000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (1 docs passed relevance criterion)
        

+ Query: query_relevance_01_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 0.5000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (1 docs passed relevance criterion)
        

+ Query: query_relevance_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 0.5000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (1 docs passed relevance criterion)
        

+ Query: query_relevance_02_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 0.7500
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (1 docs passed relevance criterion)
        

+ Query: query_relevance_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 0.5000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (1 docs passed relevance criterion)
        

+ Query: query_relevance_03_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.2500
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (0 docs passed relevance criterion)
        


</details>


### Retrieval pipeline: mpnet_base_pop_evse_v0.1.1_hybrid_search

**Test Case Performance:**



Overall pass rate: 26 out of 81 tests passed (32.1%)

Pass rates by test case name: 


- **min_pass_1**: 6 out of 10 tests passed (60.0%)


- **min_pass_2**: 0 out of 1 tests passed (0.0%)


- **one_from_hacke**: 0 out of 1 tests passed (0.0%)


- **one_from_baer**: 0 out of 1 tests passed (0.0%)


- **one_from_diehl**: 7 out of 7 tests passed (100.0%)


- **one_from_merkel**: 3 out of 3 tests passed (100.0%)


- **one_from_pickel**: 1 out of 1 tests passed (100.0%)


- **min_pass_direct_mention_1**: 3 out of 15 tests passed (20.0%)


- **min_pass_binary_relevance_3**: 5 out of 12 tests passed (41.7%)


- **one_from_baer_marandi**: 0 out of 4 tests passed (0.0%)


- **min_pass_direct_mention_3**: 0 out of 6 tests passed (0.0%)


- **exact_pass_direct_quote_1**: 0 out of 3 tests passed (0.0%)


- **min_percentage_60_scaled_relevance**: 0 out of 6 tests passed (0.0%)


- **test_case_9061e358-a8f2-47b1-9360-40c671de2faa**: 0 out of 1 tests passed (0.0%)


- **test_case_c8689144-3876-416d-a511-3d2d5945b511**: 0 out of 1 tests passed (0.0%)


- **test_case_2073fd93-dcb1-43bc-9977-bb814f5dec94**: 0 out of 1 tests passed (0.0%)


- **test_case_ba637ae3-20b0-4c7e-9483-d52b278b49e3**: 0 out of 1 tests passed (0.0%)


- **test_case_d4ce0241-a805-44cc-bf8f-1f9b15dcf053**: 0 out of 1 tests passed (0.0%)


- **test_case_6f7514a2-67af-4d3c-8624-f55d37d8f402**: 0 out of 1 tests passed (0.0%)


- **test_case_353142a0-0df4-448f-972d-9a373435cc0f**: 0 out of 1 tests passed (0.0%)


- **test_case_777d34cf-39bb-4ed5-9f0b-aa48492b96b1**: 1 out of 1 tests passed (100.0%)


- **test_case_f2915813-eeb1-4ab4-b775-62560d30b816**: 0 out of 1 tests passed (0.0%)


- **test_case_c8da6d94-2684-496a-8423-a2d579d1f0fa**: 0 out of 1 tests passed (0.0%)


- **test_case_bbc4773f-44eb-4f9a-9220-da021f7c6eb8**: 0 out of 1 tests passed (0.0%)


**Relevance Criteria Performance:**


- **direct_mention**: 24 out of 256 documents satisfied (9.4%)

- **binary_relevance**: 85 out of 256 documents satisfied (33.2%)

- **scaled_relevance**: 26 out of 256 documents satisfied (10.2%)

- **direct_quote**: 0 out of 256 documents satisfied (0.0%)



<details>
<summary>Expand/collapse query-specific details</summary>


+ Query: query_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_hacke**: ❌ FAILED (None docs passed relevance criterion)
        
        - **test_case_9061e358-a8f2-47b1-9360-40c671de2faa**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.3750
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_c8689144-3876-416d-a511-3d2d5945b511**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer**: ❌ FAILED (None docs passed relevance criterion)
        
        - **test_case_2073fd93-dcb1-43bc-9977-bb814f5dec94**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_04
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.3750
        
    + Test Case Results:
        
        - **min_pass_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_merkel**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_ba637ae3-20b0-4c7e-9483-d52b278b49e3**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_05
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 1.6250
        
    + Test Case Results:
        
        - **min_pass_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_pickel**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_d4ce0241-a805-44cc-bf8f-1f9b15dcf053**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_06
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.5000
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (2 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_6f7514a2-67af-4d3c-8624-f55d37d8f402**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_07
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.8750, Mean Score = 0.8750
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.7500
        
    + Test Case Results:
        
        - **min_pass_2**: ❌ FAILED (1 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_353142a0-0df4-448f-972d-9a373435cc0f**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_08
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_mention**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 2.7500
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (5 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_777d34cf-39bb-4ed5-9f0b-aa48492b96b1**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_09
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.2500
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_f2915813-eeb1-4ab4-b775-62560d30b816**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_10
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.8750, Mean Score = 0.8750
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.5000
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (2 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_c8da6d94-2684-496a-8423-a2d579d1f0fa**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_11
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 1.8750
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_bbc4773f-44eb-4f9a-9220-da021f7c6eb8**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_ascriptions_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_ascriptions_01_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_ascriptions_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_ascriptions_02_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_ascriptions_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.5000, Mean Score = 3.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (6 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ✅ PASSED (2 docs passed relevance criterion)
        
        - **one_from_merkel**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_ascriptions_03_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_merkel**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_paraphrase_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.2500
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_01_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.3750
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (1 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.8750
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (4 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_02_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.6250
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (4 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 2.1250
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (4 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (2 docs passed relevance criterion)
        

+ Query: query_paraphrase_03_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 2.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (5 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (1 docs passed relevance criterion)
        

+ Query: query_quotes_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_mention**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.7500, Mean Score = 3.2500
        
    + Test Case Results:
        
        - **exact_pass_direct_quote_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ✅ PASSED (5 docs passed relevance criterion)
        

+ Query: query_quotes_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.3750
        
    + Test Case Results:
        
        - **exact_pass_direct_quote_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_quotes_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.1250
        
    + Test Case Results:
        
        - **exact_pass_direct_quote_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_relevance_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.5000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_relevance_01_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_relevance_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_relevance_02_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.2500
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_relevance_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 0.5000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (1 docs passed relevance criterion)
        

+ Query: query_relevance_03_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 0.7500
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (1 docs passed relevance criterion)
        


</details>


### Retrieval pipeline: mpnet_base_pop_evse_v0.1.1_query_decomposition

**Test Case Performance:**



Overall pass rate: 44 out of 81 tests passed (54.3%)

Pass rates by test case name: 


- **min_pass_1**: 9 out of 10 tests passed (90.0%)


- **min_pass_2**: 1 out of 1 tests passed (100.0%)


- **one_from_hacke**: 1 out of 1 tests passed (100.0%)


- **one_from_baer**: 1 out of 1 tests passed (100.0%)


- **one_from_diehl**: 7 out of 7 tests passed (100.0%)


- **one_from_merkel**: 2 out of 3 tests passed (66.7%)


- **one_from_pickel**: 1 out of 1 tests passed (100.0%)


- **min_pass_direct_mention_1**: 8 out of 15 tests passed (53.3%)


- **min_pass_binary_relevance_3**: 6 out of 12 tests passed (50.0%)


- **one_from_baer_marandi**: 2 out of 4 tests passed (50.0%)


- **min_pass_direct_mention_3**: 2 out of 6 tests passed (33.3%)


- **exact_pass_direct_quote_1**: 0 out of 3 tests passed (0.0%)


- **min_percentage_60_scaled_relevance**: 0 out of 6 tests passed (0.0%)


- **test_case_9061e358-a8f2-47b1-9360-40c671de2faa**: 1 out of 1 tests passed (100.0%)


- **test_case_c8689144-3876-416d-a511-3d2d5945b511**: 1 out of 1 tests passed (100.0%)


- **test_case_2073fd93-dcb1-43bc-9977-bb814f5dec94**: 0 out of 1 tests passed (0.0%)


- **test_case_ba637ae3-20b0-4c7e-9483-d52b278b49e3**: 0 out of 1 tests passed (0.0%)


- **test_case_d4ce0241-a805-44cc-bf8f-1f9b15dcf053**: 1 out of 1 tests passed (100.0%)


- **test_case_6f7514a2-67af-4d3c-8624-f55d37d8f402**: 0 out of 1 tests passed (0.0%)


- **test_case_353142a0-0df4-448f-972d-9a373435cc0f**: 1 out of 1 tests passed (100.0%)


- **test_case_777d34cf-39bb-4ed5-9f0b-aa48492b96b1**: 0 out of 1 tests passed (0.0%)


- **test_case_f2915813-eeb1-4ab4-b775-62560d30b816**: 0 out of 1 tests passed (0.0%)


- **test_case_c8da6d94-2684-496a-8423-a2d579d1f0fa**: 0 out of 1 tests passed (0.0%)


- **test_case_bbc4773f-44eb-4f9a-9220-da021f7c6eb8**: 0 out of 1 tests passed (0.0%)


**Relevance Criteria Performance:**


- **direct_mention**: 59 out of 256 documents satisfied (23.0%)

- **binary_relevance**: 128 out of 256 documents satisfied (50.0%)

- **scaled_relevance**: 60 out of 256 documents satisfied (23.4%)

- **direct_quote**: 3 out of 256 documents satisfied (1.2%)



<details>
<summary>Expand/collapse query-specific details</summary>


+ Query: query_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_mention**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.6250, Mean Score = 2.5000
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (5 docs passed relevance criterion)
        
        - **one_from_hacke**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_9061e358-a8f2-47b1-9360-40c671de2faa**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.5000, Mean Score = 3.0000
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (2 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_c8689144-3876-416d-a511-3d2d5945b511**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.5000
        
    + Test Case Results:
        
        - **min_pass_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_2073fd93-dcb1-43bc-9977-bb814f5dec94**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_04
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.8750, Mean Score = 0.8750
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 3.0000
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_merkel**: ❌ FAILED (None docs passed relevance criterion)
        
        - **test_case_ba637ae3-20b0-4c7e-9483-d52b278b49e3**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_05
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.8750, Mean Score = 0.8750
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.1250, Mean Score = 0.1250
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.6250
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_pickel**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_d4ce0241-a805-44cc-bf8f-1f9b15dcf053**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_06
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_mention**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 2.5000
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (5 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_6f7514a2-67af-4d3c-8624-f55d37d8f402**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_07
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_mention**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.5000, Mean Score = 3.2500
        
    + Test Case Results:
        
        - **min_pass_2**: ✅ PASSED (6 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_353142a0-0df4-448f-972d-9a373435cc0f**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_08
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_mention**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 2.0000
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (3 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_777d34cf-39bb-4ed5-9f0b-aa48492b96b1**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_09
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_mention**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 2.0000
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (4 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_f2915813-eeb1-4ab4-b775-62560d30b816**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_10
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.6250
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (2 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_c8da6d94-2684-496a-8423-a2d579d1f0fa**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_11
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 2.5000
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (2 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_bbc4773f-44eb-4f9a-9220-da021f7c6eb8**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_ascriptions_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 1.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (2 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ✅ PASSED (2 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ✅ PASSED (2 docs passed relevance criterion)
        

+ Query: query_ascriptions_01_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 1.2500
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (2 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ✅ PASSED (2 docs passed relevance criterion)
        

+ Query: query_ascriptions_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.2500
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_ascriptions_02_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.1250
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_ascriptions_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (1 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_merkel**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_ascriptions_03_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (1 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_merkel**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_paraphrase_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_mention**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.6250, Mean Score = 3.3750
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (6 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ✅ PASSED (5 docs passed relevance criterion)
        

+ Query: query_paraphrase_01_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 1.0000, Mean Score = 1.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.6250, Mean Score = 3.2500
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (8 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.5000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (5 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_02_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 2.6250
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (5 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 1.0000, Mean Score = 1.0000
        
        - **direct_mention**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.5000, Mean Score = 3.5000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (8 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ✅ PASSED (5 docs passed relevance criterion)
        

+ Query: query_paraphrase_03_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 1.0000, Mean Score = 1.0000
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.5000, Mean Score = 3.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (8 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (2 docs passed relevance criterion)
        

+ Query: query_quotes_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 1.0000, Mean Score = 1.0000
        
        - **direct_mention**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 2.6250
        
    + Test Case Results:
        
        - **exact_pass_direct_quote_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ✅ PASSED (5 docs passed relevance criterion)
        

+ Query: query_quotes_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.8750, Mean Score = 0.8750
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 3.0000
        
    + Test Case Results:
        
        - **exact_pass_direct_quote_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ✅ PASSED (1 docs passed relevance criterion)
        

+ Query: query_quotes_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.8750
        
    + Test Case Results:
        
        - **exact_pass_direct_quote_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_relevance_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.2500, Mean Score = 0.2500
        
        - **scaled_relevance**: Precision = 0.2500, Mean Score = 1.0000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ✅ PASSED (2 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (2 docs passed relevance criterion)
        

+ Query: query_relevance_01_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.5000, Mean Score = 2.0000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (4 docs passed relevance criterion)
        

+ Query: query_relevance_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_mention**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.3750, Mean Score = 1.5000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ✅ PASSED (3 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (3 docs passed relevance criterion)
        

+ Query: query_relevance_02_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.3750, Mean Score = 2.2500
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (3 docs passed relevance criterion)
        

+ Query: query_relevance_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.5000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_relevance_03_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.2500
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (0 docs passed relevance criterion)
        


</details>


### Retrieval pipeline: mpnet_base_pop_evse_v0.1.1_simple_hybrid_search

**Test Case Performance:**



Overall pass rate: 23 out of 81 tests passed (28.4%)

Pass rates by test case name: 


- **min_pass_1**: 3 out of 10 tests passed (30.0%)


- **min_pass_2**: 1 out of 1 tests passed (100.0%)


- **one_from_hacke**: 0 out of 1 tests passed (0.0%)


- **one_from_baer**: 1 out of 1 tests passed (100.0%)


- **one_from_diehl**: 7 out of 7 tests passed (100.0%)


- **one_from_merkel**: 3 out of 3 tests passed (100.0%)


- **one_from_pickel**: 1 out of 1 tests passed (100.0%)


- **min_pass_direct_mention_1**: 2 out of 15 tests passed (13.3%)


- **min_pass_binary_relevance_3**: 3 out of 12 tests passed (25.0%)


- **one_from_baer_marandi**: 2 out of 4 tests passed (50.0%)


- **min_pass_direct_mention_3**: 0 out of 6 tests passed (0.0%)


- **exact_pass_direct_quote_1**: 0 out of 3 tests passed (0.0%)


- **min_percentage_60_scaled_relevance**: 0 out of 6 tests passed (0.0%)


- **test_case_9061e358-a8f2-47b1-9360-40c671de2faa**: 0 out of 1 tests passed (0.0%)


- **test_case_c8689144-3876-416d-a511-3d2d5945b511**: 0 out of 1 tests passed (0.0%)


- **test_case_2073fd93-dcb1-43bc-9977-bb814f5dec94**: 0 out of 1 tests passed (0.0%)


- **test_case_ba637ae3-20b0-4c7e-9483-d52b278b49e3**: 0 out of 1 tests passed (0.0%)


- **test_case_d4ce0241-a805-44cc-bf8f-1f9b15dcf053**: 0 out of 1 tests passed (0.0%)


- **test_case_6f7514a2-67af-4d3c-8624-f55d37d8f402**: 0 out of 1 tests passed (0.0%)


- **test_case_353142a0-0df4-448f-972d-9a373435cc0f**: 0 out of 1 tests passed (0.0%)


- **test_case_777d34cf-39bb-4ed5-9f0b-aa48492b96b1**: 0 out of 1 tests passed (0.0%)


- **test_case_f2915813-eeb1-4ab4-b775-62560d30b816**: 0 out of 1 tests passed (0.0%)


- **test_case_c8da6d94-2684-496a-8423-a2d579d1f0fa**: 0 out of 1 tests passed (0.0%)


- **test_case_bbc4773f-44eb-4f9a-9220-da021f7c6eb8**: 0 out of 1 tests passed (0.0%)


**Relevance Criteria Performance:**


- **direct_mention**: 12 out of 256 documents satisfied (4.7%)

- **binary_relevance**: 91 out of 256 documents satisfied (35.5%)

- **scaled_relevance**: 8 out of 256 documents satisfied (3.1%)

- **direct_quote**: 0 out of 256 documents satisfied (0.0%)



<details>
<summary>Expand/collapse query-specific details</summary>


+ Query: query_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_hacke**: ❌ FAILED (None docs passed relevance criterion)
        
        - **test_case_9061e358-a8f2-47b1-9360-40c671de2faa**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.3750, Mean Score = 2.6250
        
    + Test Case Results:
        
        - **min_pass_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_c8689144-3876-416d-a511-3d2d5945b511**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.5000
        
    + Test Case Results:
        
        - **min_pass_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_2073fd93-dcb1-43bc-9977-bb814f5dec94**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_04
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 2.0000
        
    + Test Case Results:
        
        - **min_pass_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_merkel**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_ba637ae3-20b0-4c7e-9483-d52b278b49e3**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_05
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.2500
        
    + Test Case Results:
        
        - **min_pass_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_pickel**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_d4ce0241-a805-44cc-bf8f-1f9b15dcf053**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_06
    + Criteria Results:
        
        - **binary_relevance**: Precision = 1.0000, Mean Score = 1.0000
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.1250, Mean Score = 2.3750
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_6f7514a2-67af-4d3c-8624-f55d37d8f402**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_07
    + Criteria Results:
        
        - **binary_relevance**: Precision = 1.0000, Mean Score = 1.0000
        
        - **direct_mention**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 2.1250
        
    + Test Case Results:
        
        - **min_pass_2**: ✅ PASSED (2 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_353142a0-0df4-448f-972d-9a373435cc0f**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_08
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.6250, Mean Score = 0.6250
        
        - **direct_mention**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 2.0000
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (3 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_777d34cf-39bb-4ed5-9f0b-aa48492b96b1**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_09
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 2.2500
        
    + Test Case Results:
        
        - **min_pass_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_f2915813-eeb1-4ab4-b775-62560d30b816**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_10
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.8750, Mean Score = 0.8750
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 2.3750
        
    + Test Case Results:
        
        - **min_pass_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_c8da6d94-2684-496a-8423-a2d579d1f0fa**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_11
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.8750, Mean Score = 0.8750
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 2.0000
        
    + Test Case Results:
        
        - **min_pass_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_diehl**: ✅ PASSED (None docs passed relevance criterion)
        
        - **test_case_bbc4773f-44eb-4f9a-9220-da021f7c6eb8**: ❌ FAILED (None docs passed relevance criterion)
        

+ Query: query_ascriptions_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.6250
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (1 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ✅ PASSED (1 docs passed relevance criterion)
        

+ Query: query_ascriptions_01_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.2500
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (1 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ✅ PASSED (1 docs passed relevance criterion)
        

+ Query: query_ascriptions_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_ascriptions_02_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_baer_marandi**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_ascriptions_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.3750
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (1 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ✅ PASSED (1 docs passed relevance criterion)
        
        - **one_from_merkel**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_ascriptions_03_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.2500, Mean Score = 0.2500
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.5000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (2 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **one_from_merkel**: ✅ PASSED (None docs passed relevance criterion)
        

+ Query: query_paraphrase_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.5000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_01_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.0000
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (1 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.6250
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (3 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_02_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.1250, Mean Score = 0.1250
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.3750
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ❌ FAILED (1 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.3750
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (4 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_paraphrase_03_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.1250
        
    + Test Case Results:
        
        - **min_pass_binary_relevance_3**: ✅ PASSED (3 docs passed relevance criterion)
        
        - **min_pass_direct_mention_3**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_quotes_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.8750, Mean Score = 0.8750
        
        - **direct_mention**: Precision = 0.5000, Mean Score = 0.5000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.5000, Mean Score = 2.8750
        
    + Test Case Results:
        
        - **exact_pass_direct_quote_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ✅ PASSED (4 docs passed relevance criterion)
        

+ Query: query_quotes_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.7500, Mean Score = 0.7500
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 2.0000
        
    + Test Case Results:
        
        - **exact_pass_direct_quote_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_quotes_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.2500
        
    + Test Case Results:
        
        - **exact_pass_direct_quote_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_relevance_01
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.3750, Mean Score = 0.3750
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 1.0000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_relevance_01_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.2500
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_relevance_02
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_relevance_02_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_relevance_03
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.2500
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (0 docs passed relevance criterion)
        

+ Query: query_relevance_03_negated
    + Criteria Results:
        
        - **binary_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_mention**: Precision = 0.0000, Mean Score = 0.0000
        
        - **direct_quote**: Precision = 0.0000, Mean Score = 0.0000
        
        - **scaled_relevance**: Precision = 0.0000, Mean Score = 0.0000
        
    + Test Case Results:
        
        - **min_pass_direct_mention_1**: ❌ FAILED (0 docs passed relevance criterion)
        
        - **min_percentage_60_scaled_relevance**: ❌ FAILED (0 docs passed relevance criterion)
        


</details>




In [2]:
l1 = [2,1,5,3]

l1.sort(key=lambda x: x, reverse=True)
print(l1)      

[5, 3, 2, 1]
